# 02 — Detección de Unauthorized Access / Elevation of Privilege con LANL
### Tema 6 — Sistemas expertos en SOC | IA Aplicada a la Ciberseguridad

**Dataset:** Comprehensive, Multi-Source Cyber-Security Events — Los Alamos National Laboratory
https://csr.lanl.gov/data/cyber1/

**Referencia académica:**
Kent, A. D. (2015). Comprehensive, Multi-Source Cyber-Security Events. Los Alamos National Laboratory.
Kent, A. D. (2016). Cyber security data sources for dynamic network research. In Dynamic Networks and
Cyber-Security, World Scientific.

**MITRE ATT&CK relacionado:** T1078 (Valid Accounts), T1021 (Remote Services — movimiento lateral),
T1548 (Abuse Elevation Control Mechanism) — https://attack.mitre.org/techniques/T1078/

## Nota importante sobre el tamaño del dataset

El dataset completo pesa ~87 GB (58 días, más de mil millones de eventos). Para un proyecto académico
**no se descarga completo**. Se recomienda:

1. Descargar únicamente `auth.txt.gz` y `redteam.txt.gz` desde https://csr.lanl.gov/data/cyber1/
2. Tomar una muestra (ej. `zcat auth.txt.gz | head -n 5000000 > auth_sample.txt`) que cubra al menos
   los primeros días, donde se concentran los eventos de redteam.
3. Documentar explícitamente en el informe que se trabajó con una muestra representativa — esta es una
   práctica común y aceptada en la literatura que usa este dataset (ver referencias arriba).

Formato de `auth.txt`: `time, source_user@domain, destination_user@domain, source_computer,
destination_computer, authentication_type, logon_type, authentication_orientation, success/failure`

Formato de `redteam.txt` (ground truth): `time, user@domain, source_computer, destination_computer`


## 1. Instalación de dependencias

In [ ]:
# !pip install scikit-learn imbalanced-learn xgboost matplotlib seaborn joblib --quiet


## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, roc_auc_score
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 3. Carga de datos (entrada del pipeline)

Ajusta las rutas según dónde hayas subido tu muestra en Kaggle/Colab.

In [ ]:
AUTH_SAMPLE_PATH = "/kaggle/input/lanl-cyber-sample/auth_sample.txt"      # AJUSTAR
REDTEAM_PATH = "/kaggle/input/lanl-cyber-sample/redteam.txt"              # AJUSTAR

auth_cols = [
    "time", "source_user_domain", "destination_user_domain",
    "source_computer", "destination_computer",
    "auth_type", "logon_type", "auth_orientation", "success",
]
redteam_cols = ["time", "user_domain", "source_computer", "destination_computer"]

auth = pd.read_csv(AUTH_SAMPLE_PATH, names=auth_cols, header=None)
redteam = pd.read_csv(REDTEAM_PATH, names=redteam_cols, header=None)

print("Auth shape:", auth.shape)
print("Redteam shape:", redteam.shape)
auth.head()


## 4. Etiquetado (join contra redteam.txt)

Un evento de auth se marca como malicioso si coincide en (usuario, computador origen, computador destino,
tiempo) con un evento del ground truth de redteam.

In [ ]:
redteam_keys = set(
    zip(redteam["time"], redteam["user_domain"], redteam["source_computer"], redteam["destination_computer"])
)

auth["is_malicious"] = auth.apply(
    lambda r: (r["time"], r["source_user_domain"], r["source_computer"], r["destination_computer"]) in redteam_keys,
    axis=1,
).astype(int)

print(auth["is_malicious"].value_counts())
print(f"Proporcion de eventos maliciosos: {auth['is_malicious'].mean():.6%}")


## 5. Feature engineering por usuario / ventana de tiempo

Los eventos crudos de auth no son buenos features por sí solos (son categóricos y de alta cardinalidad).
Se agregan por usuario en ventanas de tiempo, siguiendo el enfoque estándar en la literatura de detección
de movimiento lateral sobre este dataset (grafos de autenticación, conteo de destinos distintos, etc.).

In [ ]:
WINDOW_SECONDS = 3600  # ventana de 1 hora

auth["window"] = auth["time"] // WINDOW_SECONDS

grouped = auth.groupby(["source_user_domain", "window"]).agg(
    n_events=("time", "count"),
    n_distinct_dest_computers=("destination_computer", "nunique"),
    n_distinct_source_computers=("source_computer", "nunique"),
    n_failed=("success", lambda s: (s == "Fail").sum()),
    n_success=("success", lambda s: (s == "Success").sum()),
    n_logon_types=("logon_type", "nunique"),
    n_auth_types=("auth_type", "nunique"),
    label=("is_malicious", "max"),
).reset_index()

grouped["fail_ratio"] = grouped["n_failed"] / grouped["n_events"].replace(0, 1)

print(grouped.shape)
print(grouped["label"].value_counts())
grouped.head()


## 6. Split, escalado y balanceo (SMOTE)

Este dataset es extremadamente desbalanceado (~0.0001% de eventos maliciosos) — SMOTE y class_weight son obligatorios, no opcionales.

In [ ]:
feature_cols = [
    "n_events", "n_distinct_dest_computers", "n_distinct_source_computers",
    "n_failed", "n_success", "n_logon_types", "n_auth_types", "fail_ratio",
]

X = grouped[feature_cols]
y = grouped["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, y_train.sum() - 1) if y_train.sum() > 1 else 1)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print("Original:", np.bincount(y_train))
print("Balanceado:", np.bincount(y_train_bal))


## 7. Entrenamiento (XGBoost con peso de clase adicional)

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="aucpr",   # mejor que logloss para clases muy desbalanceadas
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train_bal, y_train_bal)


## 8. Evaluación

En datasets extremadamente desbalanceados, prioriza recall y AUC-PR sobre precisión global — un accuracy alto es engañoso aquí (el modelo trivial que siempre predice "benigno" ya tiene >99.99% de accuracy).

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Benigno", "Malicioso"]))

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")
auc = roc_auc_score(y_test, y_proba)

metrics_table = pd.DataFrame({
    "Metrica": ["Precision", "Recall", "F1-score", "AUC"],
    "Valor": [round(precision, 4), round(recall, 4), round(f1, 4), round(auc, 4)],
})
metrics_table


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Benigno", "Malicioso"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Purples", colorbar=True)
plt.title("Matriz de confusion - Unauthorized Access / Privilege Escalation (LANL)")
plt.tight_layout()
plt.savefig("privesc_confusion_matrix.png", dpi=150)
plt.show()


## 9. Feature importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(7, 5))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.xlabel("Importancia")
plt.title("Importancia de features - Deteccion de accesos no autorizados")
plt.tight_layout()
plt.savefig("privesc_feature_importance.png", dpi=150)
plt.show()

importances


## 10. Guardado del modelo (usado por el notebook 03 de fusión)

In [ ]:
joblib.dump(model, "privesc_model.joblib")
joblib.dump(scaler, "privesc_scaler.joblib")
joblib.dump(feature_cols, "privesc_feature_cols.joblib")

print("Modelo de privilege escalation guardado. Copialo junto al notebook 03 antes de ejecutarlo.")
